# Receiving CHIME/FRB Alerts via NASA GCN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amozdan/sharing-the-sky/blob/main/chime_frb_gcn_alerts.ipynb)

This notebook walks through setting up a Python consumer to receive real-time CHIME/FRB Fast Radio Burst alerts from [NASA's General Coordinates Network (GCN)](https://gcn.nasa.gov).

GCN is the global alert platform used by the multi-messenger astronomy community for gravitational waves, gamma-ray bursts, neutrinos — and now FRBs. CHIME/FRB joined GCN in February 2026, making its detections available to any subscribed observatory with no additional setup.

---

## 1. Create a GCN Account and Get Credentials

1. Go to **[gcn.nasa.gov](https://gcn.nasa.gov)** and sign up for a free account
2. Once logged in, navigate to your account menu (top right) → **Credentials**
3. Click **New Credentials**
   - Under **Scope**, select `gcn.nasa.gov/kafka-public-consumer`
   - Under **Customize Alerts**, you can optionally pre-select notice types (we'll subscribe manually in code)
4. Copy your `client_id` and `client_secret` — they are only shown once

> ⚠️ **Keep your credentials private.** Do not commit them to a public repository.

> ⚠️ **Credentials expire after 30 days of inactivity.** GCN will email you a reminder. Simply reconnecting resets the timer.

---

## 2. Install the GCN Kafka Client

In [ ]:
# Install the gcn-kafka Python client
# Only needs to be run once
%pip install gcn-kafka

## 3. Set Your Credentials

Paste your `client_id` and `client_secret` below. For production use, consider storing them in environment variables or a `.env` file rather than hardcoding them.

In [ ]:
CLIENT_ID     = 'YOUR_CLIENT_ID'      # replace with your GCN client_id
CLIENT_SECRET = 'YOUR_CLIENT_SECRET'  # replace with your GCN client_secret

## 4. Listen for New Alerts (Live Mode)

This consumer starts listening from **now** — it will only receive alerts that arrive after you start it. Run this cell and leave it running; each new CHIME/FRB detection will print as it comes in.

CHIME/FRB detects ~600 FRBs per year (~1–2 per day), so you may need to wait a while for a new detection. Interrupt the cell (■ stop button) when done.

In [ ]:
import json
from gcn_kafka import Consumer

consumer = Consumer(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    domain='gcn.nasa.gov'
    # No auto.offset.reset set — defaults to 'latest' (new alerts only)
)

consumer.subscribe(['gcn.notices.chime.frb'])

print('Listening for CHIME/FRB alerts... (interrupt to stop)')

try:
    while True:
        for message in consumer.consume(timeout=1):
            if message.error():
                print('Error:', message.error())
                continue

            notice = json.loads(message.value())

            print('─' * 60)
            print(f"Topic:      {message.topic()}")
            print(f"Offset:     {message.offset()}")
            print(f"Alert type: {notice.get('alert_type', 'N/A')}")
            print(f"Event ID:   {notice.get('id', 'N/A')}")
            print(f"Trigger time: {notice.get('trigger_time', 'N/A')}")
            print(f"RA / Dec:   {notice.get('ra', 'N/A')} / {notice.get('dec', 'N/A')}")
            print(f"DM:         {notice.get('dm', 'N/A')} pc/cm³")
            print(f"SNR:        {notice.get('snr', 'N/A')}")
            print(f"Importance: {notice.get('importance', 'N/A')}")
            print()

except KeyboardInterrupt:
    print('Stopped.')
finally:
    consumer.close()

## 5. Replay Past Alerts (Archive Mode)

Setting `auto.offset.reset` to `'earliest'` replays **all** CHIME/FRB notices since GCN launch (February 2026). This is useful for testing your pipeline or building a local archive.

> ⚠️ This will stream a large number of messages (~600+ per year). Add a counter or break condition if you only want a sample.

In [ ]:
import json
from gcn_kafka import Consumer

MAX_ALERTS = 10  # set to None to receive all

consumer = Consumer(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    domain='gcn.nasa.gov',
    config={'auto.offset.reset': 'earliest'}  # replay from the beginning
)

consumer.subscribe(['gcn.notices.chime.frb'])

print(f'Replaying past CHIME/FRB alerts (max={MAX_ALERTS})...')
count = 0

try:
    while True:
        for message in consumer.consume(timeout=5):
            if message.error():
                print('Error:', message.error())
                continue

            notice = json.loads(message.value())
            count += 1

            print('─' * 60)
            print(f"[{count}] Offset: {message.offset()}")
            print(f"Alert type:   {notice.get('alert_type', 'N/A')}")
            print(f"Event ID:     {notice.get('id', 'N/A')}")
            print(f"Trigger time: {notice.get('trigger_time', 'N/A')}")
            print(f"RA / Dec:     {notice.get('ra', 'N/A')} / {notice.get('dec', 'N/A')}")
            print(f"DM:           {notice.get('dm', 'N/A')} pc/cm³")
            print(f"SNR:          {notice.get('snr', 'N/A')}")
            print(f"Importance:   {notice.get('importance', 'N/A')}")
            print()

            if MAX_ALERTS and count >= MAX_ALERTS:
                print(f'Reached MAX_ALERTS={MAX_ALERTS}. Set MAX_ALERTS=None to receive all.')
                raise KeyboardInterrupt

except KeyboardInterrupt:
    print(f'Stopped after {count} alerts.')
finally:
    consumer.close()

## 6. Replay from a Specific Date

To start from a specific date rather than the very beginning, use `seek_to_time()` after subscribing. This is useful if you want to catch up from a known point without replaying everything.

In [ ]:
import json
from datetime import datetime, timezone
from gcn_kafka import Consumer

# Set the start date (UTC)
START_DATE = datetime(2026, 4, 1, tzinfo=timezone.utc)
MAX_ALERTS = 5

consumer = Consumer(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    domain='gcn.nasa.gov',
    config={'auto.offset.reset': 'earliest'}
)

topic = 'gcn.notices.chime.frb'
consumer.subscribe([topic])

# Seek to the start date
start_ms = int(START_DATE.timestamp() * 1000)  # Kafka uses milliseconds
consumer.seek_to_time(start_ms)

print(f'Replaying CHIME/FRB alerts from {START_DATE.date()} (max={MAX_ALERTS})...')
count = 0

try:
    while True:
        for message in consumer.consume(timeout=5):
            if message.error():
                print('Error:', message.error())
                continue

            notice = json.loads(message.value())
            count += 1

            print('─' * 60)
            print(f"[{count}] Offset: {message.offset()}")
            print(f"Alert type:   {notice.get('alert_type', 'N/A')}")
            print(f"Event ID:     {notice.get('id', 'N/A')}")
            print(f"Trigger time: {notice.get('trigger_time', 'N/A')}")
            print(f"RA / Dec:     {notice.get('ra', 'N/A')} / {notice.get('dec', 'N/A')}")
            print(f"DM:           {notice.get('dm', 'N/A')} pc/cm³")
            print(f"SNR:          {notice.get('snr', 'N/A')}")
            print(f"Importance:   {notice.get('importance', 'N/A')}")
            print()

            if MAX_ALERTS and count >= MAX_ALERTS:
                raise KeyboardInterrupt

except KeyboardInterrupt:
    print(f'Stopped after {count} alerts.')
finally:
    consumer.close()

## 7. Understanding the Notice Schema

Each CHIME/FRB GCN notice is a JSON payload with the following key fields:

| Field | Description |
|---|---|
| `alert_type` | `DETECTION`, `SUBSEQUENT`, `UPDATE`, or `RETRACTION` |
| `id` | Unique event identifier |
| `trigger_time` | UTC time of the detection (ISO 8601) |
| `ra` / `dec` | Sky position in degrees (J2000) |
| `ra_uncertainty` / `dec_uncertainty` | Positional uncertainty in degrees |
| `dm` | Dispersion measure in pc/cm³ |
| `dm_uncertainty` | DM uncertainty |
| `snr` | Signal-to-noise ratio |
| `importance` | ML-based importance score (0–1); higher = more likely astrophysical |

Full schema documentation: [gcn.nasa.gov/missions/chime](https://gcn.nasa.gov/missions/chime)

---

## 8. Further Resources

- [CHIME/FRB GCN Mission Page](https://gcn.nasa.gov/missions/chime) — schema, subscription, examples
- [GCN Kafka Client Setup](https://gcn.nasa.gov/docs/client) — official documentation
- [gcn-kafka Python package](https://github.com/nasa-gcn/gcn-kafka-python) — source and examples
- [CHIME/FRB Open Data](https://chime-frb-open-data.github.io) — public data releases
- [VOEvent alerts](https://chime-frb-open-data.github.io/voevents/) — the original CHIME/FRB alert stream